# 🏏 Member 1: Sampling Frame Construction & Simple Random Sampling (SRSWOR vs SRSWR)
**Project:** Design, Estimation and Efficiency Comparison of Sampling Techniques on IPL Batting Data (2008–2024)

### Objectives:
1. Construct the unified sampling frame from raw deliveries and match records (innings 1 & 2).
2. Clean and canonicalise franchise names, season identifiers, and venue names.
3. Verify target population metrics: $N = 16,515$, $\mu = 19.97$, $\sigma = 21.35$, $P(\\text{runs} \\ge 30) = 0.2488$.
4. Calculate sample size $n = 1,101$ with finite population correction (FPC).
5. Draw and compare Simple Random Sampling Without Replacement (SRSWOR) and With Replacement (SRSWR).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from sampling.frame import build_frame, load_frame
from sampling import srs, estimators as est

print(f"Configured N: {config.N:,}, Common sample size n: {config.n:,}")


## 1. Frame Construction and Data Quality Verification


In [ ]:
# Build or load the cleaned frame
frame, quality_log = build_frame(save=False)

print("Frame Shape:", frame.shape)
print("Unique Matches:", frame["match_id"].nunique())
print("True Mean Runs:", round(frame["runs"].mean(), 4))
print("True SD Runs (pop):", round(frame["runs"].std(ddof=0), 4))
print("True Proportion >= 30:", round((frame["runs"] >= 30).mean(), 4))
frame.head()


## 2. Sample Size Determination (Justifying $n = 1,101$)
Using Cochran's formula for a finite population:
$$n_0 = \left( \frac{z \cdot S}{e} \right)^2, \quad n = \frac{n_0}{1 + n_0 / N}$$
At $z = 1.96$ and $S \approx 21.35$, setting margin $e = 1.22$ yields $n \approx 1,101$.


In [ ]:
S = float(frame["runs"].std(ddof=1))
P = float((frame["runs"] >= 30).mean())

justification = srs.sample_size_justification(S, P, N=len(frame))
for k, v in justification.items():
    print(f"{k}: {v}")


## 3. SRSWOR vs SRSWR Estimation & Theoretical Ratio
Textbook variance ratio:
$$\frac{V(\bar{y}_{\text{WOR}})}{V(\bar{y}_{\text{WR}})} = \frac{N - n}{N - 1} \approx 1 - f$$


In [ ]:
comp_srs = srs.compare_wor_wr(frame, rng=np.random.default_rng(config.SEED))

print("=== SRSWOR Results ===")
for k in ["mean", "se_mean", "ci_mean", "var_mean", "fpc", "abs_error_mean"]:
    print(f"  {k}: {comp_srs['srswor'][k]}")

print("\n=== SRSWR Results ===")
for k in ["mean", "se_mean", "ci_mean", "var_mean", "abs_error_mean"]:
    print(f"  {k}: {comp_srs['srswr'][k]}")

print(f"\nTheoretical Variance Ratio (N-n)/(N-1): {comp_srs['var_ratio_theory']:.6f}")
print(f"Observed Variance Ratio: {comp_srs['var_ratio_observed']:.6f}")


## 4. Visualising the Population Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(frame["runs"], bins=50, color=config.PALETTE[0], edgecolor="white")
axes[0].axvline(frame["runs"].mean(), color="red", linestyle="--", label=f"Mean: {frame['runs'].mean():.2f}")
axes[0].set_title("Distribution of Runs per Batter-Innings")
axes[0].set_xlabel("Runs")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Boxplot by stratum
sns.boxplot(data=frame, x="stratum", y="runs", order=config.STRATA_LABELS, palette=config.PALETTE[:4], ax=axes[1])
axes[1].set_title("Runs by Batting Position Stratum")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()
